In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pickle
import scipy.stats

# dbfile_1 = open(
#     "/Users/michael/Data/Junk_data/coexpression_matrix_From_infected_data_full_dataset.p",
#     "ab",
# )
# pickle.dump(final_result, dbfile_1)
# dbfile_1.close()
dbfile_1 = open(
    "/Users/michael/Data/Junk_data/coexpression_matrix_From_infected_data.p", "rb"
)
final_result = pickle.load(dbfile_1)

In [3]:
final_result

,AT1G01010,AT1G01020,AT1G03987,AT1G01030,AT1G01040,AT1G01046,AT1G01050,AT1G03997,AT1G01060,AT1G01070,...,ArthCp081,Arthcp087,ArthCr091,ArthCr089,ArthCr088,ArthCp088,ArthCp086,ArthCp083,ArthCp084,ArthCp085
AT1G01010,1.000000,0.636791,0.553707,0.531361,0.669665,0.698443,0.473663,0.240847,-0.008590,0.467666,...,-0.134841,-0.165471,0.175263,0.326686,0.218201,-0.096639,0.135294,0.104379,-0.129141,0.000546
AT1G01020,0.636791,1.000000,0.450511,0.672609,0.790162,0.761237,0.792788,0.301020,-0.200810,0.279741,...,0.215550,0.247964,-0.046122,0.258916,0.124026,0.212605,0.045098,0.204014,-0.106080,0.015829
AT1G01030,0.531361,0.672609,0.499148,1.000000,0.548924,0.460693,0.630065,0.064470,-0.190843,0.176094,...,0.284429,0.332401,0.239834,0.330896,0.252317,0.173950,-0.109524,0.213503,-0.202936,0.013100
AT1G01040,0.669665,0.790162,0.626004,0.548924,1.000000,0.844441,0.521049,0.187513,0.182002,0.511730,...,0.136537,0.025849,-0.083016,0.249435,0.224588,0.006442,0.238366,0.017080,-0.036896,0.040390
AT1G01046,0.698443,0.761237,0.641306,0.460693,0.844441,1.000000,0.486492,0.250046,0.043445,0.429418,...,-0.045917,-0.124691,-0.110818,0.207714,0.077856,-0.064499,0.264446,0.028974,-0.156993,0.084153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
unassigned_gene_9,0.065093,-0.030868,0.168110,-0.047001,0.134844,0.045981,-0.192949,-0.246206,0.304391,0.272590,...,-0.048212,-0.116939,0.191076,-0.043015,0.341075,-0.118625,0.222422,-0.162706,-0.169845,-0.066582
unassigned_gene_94,0.354365,0.264518,0.367220,0.126729,0.308790,0.425130,-0.057019,0.000952,-0.057812,0.226447,...,-0.362635,-0.385988,-0.040126,0.152849,0.132303,-0.147134,0.287261,-0.109385,-0.210663,-0.201822
unassigned_gene_96,-0.393969,-0.340307,-0.147011,-0.283575,-0.325743,-0.411005,-0.164350,-0.388598,-0.011607,-0.114725,...,0.137513,0.108749,0.086167,0.012451,0.019119,-0.187231,0.127050,-0.088146,0.181908,0.128034
unassigned_gene_97,-0.234515,-0.125249,-0.224401,-0.301318,-0.111933,-0.075196,-0.003887,-0.032734,0.329885,-0.019142,...,0.127576,0.085796,-0.154847,-0.224803,0.001065,-0.108150,0.270375,-0.211628,0.044242,0.115843


In [6]:
def Calculate_Score_list_for_thresholding(
    network_1,
    network_2,
    cross_species_n_m_genes_path,
    Species_1="rice",
    Species_2="maize",
):
    import pandas as pd

    # Get Species Names in Common form
    common_name_1 = Name_resolver.species_name_resolver(Species_1, "common")
    common_name_2 = Name_resolver.species_name_resolver(Species_2, "common")
    if common_name_1 == "rice":
        common_name_1 = "rice_jp"
    if common_name_2 == "rice":
        common_name_2 = "rice_jp"

    cross_species_n_m_genes = pd.read_csv(cross_species_n_m_genes_path)
    orig_column_common_name_1 = common_name_1 + " Symbol"
    orig_column_common_name_2 = common_name_2 + " Symbol"
    cross_species_n_m_genes = cross_species_n_m_genes.rename(
        columns={
            orig_column_common_name_1: common_name_1,
            orig_column_common_name_2: common_name_2,
        }
    )
    ### Get one to ones
    cross_species_map_one_to_one = cross_species_n_m_genes.drop_duplicates(
        subset=common_name_1,
        keep=False,
    )
    cross_species_map_one_to_one = cross_species_map_one_to_one.drop_duplicates(
        subset=common_name_2, keep=False
    )

    ## Convert to Dictionary
    dictionary_mapper_one_to_two = cross_species_map_one_to_one.set_index(
        common_name_1
    ).to_dict()[common_name_2]
    dictionary_mapper_dos_to_uno = cross_species_map_one_to_one.set_index(
        common_name_2
    ).to_dict()[common_name_1]

    ## Read In Cococonets
    coconet_species_one = network_1
    coconet_species_two = network_2

    cross_species_n_m_genes["Group ID"] = "Unassigned"

    ## Assign Genes to Groups
    id_indexer = 0
    for gene_pair in cross_species_n_m_genes.iterrows():

        if gene_pair[1]["Group ID"] == "Unassigned":
            current_species_1_gene = gene_pair[1][common_name_1]
            current_species_2_gene = gene_pair[1][common_name_2]
            cross_species_n_m_genes["Group ID"].loc[
                (cross_species_n_m_genes[common_name_1] == current_species_1_gene)
                & (cross_species_n_m_genes["Group ID"] == "Unassigned")
            ] = id_indexer
            cross_species_n_m_genes["Group ID"].loc[
                (cross_species_n_m_genes[common_name_2] == current_species_2_gene)
                & (cross_species_n_m_genes["Group ID"] == "Unassigned")
            ] = id_indexer

            all_labeled_groups = cross_species_n_m_genes.loc[
                cross_species_n_m_genes["Group ID"] == id_indexer
            ]

            all_labeled_groups_species_1_genes = all_labeled_groups[
                common_name_1
            ].to_list()
            all_labeled_groups_species_2_genes = all_labeled_groups[
                common_name_2
            ].to_list()

            cross_species_n_m_genes["Group ID"].loc[
                cross_species_n_m_genes[common_name_1].isin(
                    all_labeled_groups_species_1_genes
                )
            ] = id_indexer
            cross_species_n_m_genes["Group ID"].loc[
                cross_species_n_m_genes[common_name_2].isin(
                    all_labeled_groups_species_2_genes
                )
            ] = id_indexer

            id_indexer += 1

    # Identify Pairs for evaluation
    all_pairs_to_evaluate_for_functional_conservation = pd.DataFrame(
        columns=[common_name_1, common_name_2, "Group Number"]
    )
    for group_number in list(set(cross_species_n_m_genes["Group ID"].to_list())):
        current_gene_map = cross_species_n_m_genes.loc[
            cross_species_n_m_genes["Group ID"] == group_number
        ]
        list_of_species_1_genes_in_group = list(
            set(current_gene_map[common_name_1].to_list())
        )
        list_of_species_2_genes_in_group = list(
            set(current_gene_map[common_name_2].to_list())
        )
        all_combo_list_current_genes = itertools.product(
            list_of_species_1_genes_in_group, list_of_species_2_genes_in_group
        )
        all_combo_list_current_genes = list(map(list, all_combo_list_current_genes))
        current_list_of_pairs = pd.DataFrame(
            all_combo_list_current_genes, columns=[common_name_1, common_name_2]
        )
        current_list_of_pairs["Group Number"] = group_number
        all_pairs_to_evaluate_for_functional_conservation = (
            all_pairs_to_evaluate_for_functional_conservation.append(
                current_list_of_pairs
            )
        )

    all_pairs_to_evaluate_for_functional_conservation["Species 1 Score"] = np.nan
    all_pairs_to_evaluate_for_functional_conservation["Species 2 Score"] = np.nan

    ## Trim cococonets to match

    trimmed_species_1_cococonet = coconet_species_one[
        coconet_species_one.columns.intersection(
            cross_species_n_m_genes[common_name_1].to_list()
        )
    ]
    trimmed_species_1_cococonet = trimmed_species_1_cococonet[
        trimmed_species_1_cococonet.index.isin(
            cross_species_n_m_genes[common_name_1].to_list()
        )
    ]
    double_species_1_trimmed_cococonet = trimmed_species_1_cococonet[
        trimmed_species_1_cococonet.columns.intersection(
            cross_species_map_one_to_one[common_name_1].to_list()
        )
    ]
    double_species_1_trimmed_cococonet = double_species_1_trimmed_cococonet.replace(
        1, 0
    )

    trimmed_species_2_cococonet = coconet_species_two[
        coconet_species_two.columns.intersection(
            cross_species_n_m_genes[common_name_2].to_list()
        )
    ]
    trimmed_species_2_cococonet = trimmed_species_2_cococonet[
        trimmed_species_2_cococonet.index.isin(
            cross_species_n_m_genes[common_name_2].to_list()
        )
    ]
    double_species_2_trimmed_cococonet = trimmed_species_2_cococonet[
        trimmed_species_2_cococonet.columns.intersection(
            cross_species_map_one_to_one[common_name_2].to_list()
        )
    ]
    double_species_2_trimmed_cococonet = double_species_2_trimmed_cococonet.replace(
        1, 0
    )

    ## Rank
    species_1_cococonet_ranked = trimmed_species_1_cococonet.rank()
    species_2_cococonet_ranked = trimmed_species_2_cococonet.rank()

    # Do top 10 Genes
    top_10_species_1_genes = np.array(
        [
            double_species_1_trimmed_cococonet.T[c].nlargest(10).index.values
            for c in double_species_1_trimmed_cococonet.T
        ]
    )  # using pair list above, cut down top 10 list to relevant genes, probably by adding list as a column in panda and then filtering panda to index of pair list
    top_10_species_1_genes_dataframe = pd.DataFrame(
        data=top_10_species_1_genes,
        index=double_species_1_trimmed_cococonet.index,
        columns=[
            "One",
            "Two",
            "Three",
            "Four",
            "Five",
            "Six",
            "Seven",
            "Eight",
            "Nine",
            "Ten",
        ],
    )

    # Convert
    top_10_species_1_genes_as_species_2 = top_10_species_1_genes_dataframe.replace(
        to_replace=dictionary_mapper_one_to_two
    )

    # Get genes for checking
    have_species_1_pairs = all_pairs_to_evaluate_for_functional_conservation.loc[
        all_pairs_to_evaluate_for_functional_conservation[common_name_1].isin(
            top_10_species_1_genes_as_species_2.index
        )
    ]
    trimmed_all_gene_pairs_for_fc = have_species_1_pairs.loc[
        have_species_1_pairs[common_name_2].isin(trimmed_species_2_cococonet.index)
    ]
    trimmed_all_gene_pairs_for_fc = trimmed_all_gene_pairs_for_fc.reset_index(drop=True)

    # Get values in species 2
    for two_genes in trimmed_all_gene_pairs_for_fc.iterrows():
        current_species_1_gene = two_genes[1][common_name_1]
        current_species_2_gene = two_genes[1][common_name_2]
        finger_print_genes = top_10_species_1_genes_as_species_2.loc[
            current_species_1_gene
        ].to_list()
        gene_ranks_in_species_2 = species_2_cococonet_ranked.loc[
            species_2_cococonet_ranked.index.isin(finger_print_genes),
            current_species_2_gene,
        ]
        avg_rank_in_species_2 = gene_ranks_in_species_2.mean()
        index_from_pairs = two_genes[0]
        trimmed_all_gene_pairs_for_fc.at[index_from_pairs, "Species 1 Score"] = (
            avg_rank_in_species_2
        )

    # Repeat for Species 2

    top_10_species_2_genes = np.array(
        [
            double_species_2_trimmed_cococonet.T[c].nlargest(10).index.values
            for c in double_species_2_trimmed_cococonet.T
        ]
    )  # using pair list above, cut down top 10 list to relevant genes, probably by adding list as a column in panda and then filtering panda to index of pair list
    top_10_species_2_genes_dataframe = pd.DataFrame(
        data=top_10_species_2_genes,
        index=double_species_2_trimmed_cococonet.index,
        columns=[
            "One",
            "Two",
            "Three",
            "Four",
            "Five",
            "Six",
            "Seven",
            "Eight",
            "Nine",
            "Ten",
        ],
    )

    # convert
    top_10_species_2_genes_as_species_1 = top_10_species_2_genes_dataframe.replace(
        to_replace=dictionary_mapper_dos_to_uno
    )

    # Get values in species 1
    for two_genes in trimmed_all_gene_pairs_for_fc.iterrows():
        current_species_1_gene = two_genes[1][common_name_1]
        current_species_2_gene = two_genes[1][common_name_2]
        finger_print_genes = top_10_species_2_genes_as_species_1.loc[
            current_species_2_gene
        ].to_list()
        gene_ranks_in_species_1 = species_1_cococonet_ranked.loc[
            species_1_cococonet_ranked.index.isin(finger_print_genes),
            current_species_1_gene,
        ]
        avg_rank_in_species_1 = gene_ranks_in_species_1.mean()
        index_from_pairs = two_genes[0]
        trimmed_all_gene_pairs_for_fc.loc[index_from_pairs, "Species 2 Score"] = (
            avg_rank_in_species_1
        )

    # Caluclate Divisors
    Number_of_species_1_genes = len(top_10_species_1_genes_as_species_2)
    Number_of_species_2_genes = len(top_10_species_2_genes_as_species_1)

    species_1_score_divisor = Number_of_species_2_genes - 4.5
    species_2_score_divisor = Number_of_species_1_genes - 4.5

    # Divide and Average
    trimmed_all_gene_pairs_for_fc["Species 1 Score"] = (
        trimmed_all_gene_pairs_for_fc["Species 1 Score"] / species_1_score_divisor
    )
    trimmed_all_gene_pairs_for_fc["Species 2 Score"] = (
        trimmed_all_gene_pairs_for_fc["Species 2 Score"] / species_2_score_divisor
    )
    trimmed_all_gene_pairs_for_fc["Total Score"] = trimmed_all_gene_pairs_for_fc[
        ["Species 1 Score", "Species 2 Score"]
    ].mean(axis=1)

    return trimmed_all_gene_pairs_for_fc